# Data exploration checks

This notebook reads `data/ir_registration_synthetic.csv` as synthetic data and runs the checks documented in `data-exploration.md`. It does not write to the source CSV.

In [ ]:
import csv
from collections import Counter, defaultdict
from pathlib import Path

source_path = Path('data/ir_registration_synthetic.csv')
with source_path.open('r', encoding='utf-8-sig', newline='') as handle:
    reader = csv.DictReader(handle)
    fieldnames = reader.fieldnames or []
    rows = list(reader)

count_fields = ['報名人數', '錄取人數', '報到人數', '註冊人數', '休學人數']
key_fields = ['學年度', '系所', '招生管道']

missing_by_column = {
    field: sum(row.get(field, '').strip() == '' for row in rows)
    for field in fieldnames
}
exact_row_counts = Counter(tuple(row[field] for field in fieldnames) for row in rows)
key_counts = Counter(tuple(row[field] for field in key_fields) for row in rows)

logic_violations = {
    'applicant_chain': [],
    'leave_over_registered': [],
    'negative_count': [],
}
for csv_line, row in enumerate(rows, start=2):
    values = {field: int(row[field]) for field in count_fields}
    if not (values['報名人數'] >= values['錄取人數'] >= values['報到人數'] >= values['註冊人數']):
        logic_violations['applicant_chain'].append(csv_line)
    if values['休學人數'] > values['註冊人數']:
        logic_violations['leave_over_registered'].append(csv_line)
    for field, value in values.items():
        if value < 0:
            logic_violations['negative_count'].append((csv_line, field))

zero_denominators = {
    field: [csv_line for csv_line, row in enumerate(rows, start=2) if int(row[field]) == 0]
    for field in ['錄取人數', '報到人數', '註冊人數']
}
admitted_by_department_channel = defaultdict(set)
for row in rows:
    admitted_by_department_channel[(row['系所'], row['招生管道'])].add(int(row['錄取人數']))

def summarize(group_fields):
    summary = defaultdict(lambda: {'筆數': 0, **{field: 0 for field in count_fields}})
    for row in rows:
        group = tuple(row[field] for field in group_fields)
        summary[group]['筆數'] += 1
        for field in count_fields:
            summary[group][field] += int(row[field])
    return dict(summary)

checks = {
    'source_path': str(source_path.resolve()),
    'row_count': len(rows),
    'column_count': len(fieldnames),
    'fieldnames': fieldnames,
    'missing_by_column': missing_by_column,
    'exact_duplicate_groups': sum(count > 1 for count in exact_row_counts.values()),
    'composite_key_duplicate_groups': sum(count > 1 for count in key_counts.values()),
    'logic_violations': logic_violations,
    'zero_denominators': zero_denominators,
    'admitted_not_fixed_groups': sum(len(values) > 1 for values in admitted_by_department_channel.values()),
    'by_year': summarize(['學年度']),
    'by_department': summarize(['系所']),
    'by_channel': summarize(['招生管道']),
}
checks